In [1]:
from sage.libs.ecl import ecl_eval
ecl_eval("(ext:set-limit 'ext:heap-size 0)")   # 0 = no ECL-side limit

<ECL: 0>

In [2]:
var('n e_t u_bar v_bar f_4t f_5t g_4t g_5t Xi dt du dv gamma F_phi e_phi y K G m')
var('u_bar m G eta gamma e_t n Xi')
var('y y_0 h TE y_1PN y_15PN y_2PN y_25PN Xi chi_A chi_S delta omega')
var('e_t gamma y_inv e0 omega_dash A B flagTail e0 x0 flagHF SO chi w0 f0 j f_var f0_var e y0 kappaA kappaS horiz z')

(e_t,
 gamma,
 y_inv,
 e0,
 omega_dash,
 A,
 B,
 flagTail,
 e0,
 x0,
 flagHF,
 SO,
 chi,
 w0,
 f0,
 j,
 f_var,
 f0_var,
 e,
 y0,
 kappaA,
 kappaS,
 horiz,
 z)

In [3]:
# e_t(e0, chi) at all PN orders (Xi^0 = paper E_0)
import re
filename = "../et_e0/NNLO_e0_faster/et_e0_NNLO_3PN.txt"
with open(filename, "r") as f:
    content = f.read()
match = re.search(r"et_e0_NNLO_3PN\s*=\s*(.*)", content, re.DOTALL)
if match:
    expr_str = match.group(1).strip()
    et_e0_NNLO_3PN = (sage_eval(expr_str, locals=globals())).expand().subs({SO^2:0},{SO^3:0})

In [4]:
# e_t(e0, x, x0) to e0^3, Xi^0..Xi^4 (enough for k to 3PN); chi = (x/x0)^(3/2)
W=SR.wild(0)
et_x_expr=sum((et_e0_NNLO_3PN.coefficient(Xi,p).taylor(e0,0,3)).expand().subs({chi^W:x^(3/2*W)*x0^(-3/2*W)})*Xi^p for p in range(5)).expand()

In [ ]:
# k/(1+k) per PN order in x and e_t (K_files/k_xe.txt: Henry & Khalil SO + MGS nonspinning 3PN, our symbols)
filename = "K_files/k_xe.txt"
with open(filename, "r") as f:
    content = f.read()
match = re.search(r"k_xe\s*=\s*(.*)", content, re.DOTALL)
if match:
    k_xe = sage_eval(match.group(1).strip(), locals=globals())

# Xi^p part of k/(1+k) with e_t -> e_t(e0,x,x0) at all PN orders, to e0^4
def k_PN(p):
    k_e = sum(k_xe.coefficient(Xi,q).taylor(e_t,0,4)*Xi^q for q in range(2,p+1))
    return ((k_e.subs({e_t:et_x_expr}).expand().coefficient(Xi,p)).taylor(e0,0,4)).expand()

k_names = {2:"k_1PN_expr", 3:"k_15PN_expr", 4:"k_2PN_expr", 5:"k_25PN_expr", 6:"k_3PN_expr"}
k_exprs = {name: k_PN(p) for p, name in k_names.items()}

In [ ]:
# one txt per PN order, same format as the other files
for name, ex in k_exprs.items():
    with open("K_files/%s.txt" % name, "w") as f:
        f.write("# %s: Xi^p part of k/(1+k) with e_t(e0,x,x0) substituted, to e0^4\n" % name)
        f.write("# Plain ASCII output from Sage\n\n")
        f.write("%s =\n%s\n\n" % (name, ex))